# QuantRisk Analytics: exploratory analysis

This notebook calls the same `src/` functions as the dashboard to study a
sample portfolio in more depth than the UI shows.

> **Data notice:** it uses the bundled **synthetic** dataset
> (`data/sample_prices.csv`, `SYN_` tickers). The numbers illustrate the
> methodology and say nothing about real markets. This is not investment advice.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.analysis import AnalysisConfig, run_analysis
from src.data_loader import compute_returns, load_sample_prices
from src.performance import annualized_return, annualized_volatility, max_drawdown, sharpe_ratio
from src.portfolio import construct_portfolio
from src.risk_metrics import historical_expected_shortfall, historical_var, var_es_table

pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.width", 140)

## 1. Data and data quality
Observation counts, first and last price per asset. The gaps are deliberate (see `data/README.md`).

In [2]:
prices = load_sample_prices()
pd.DataFrame({
    "first_price": prices.apply(lambda s: s.first_valid_index().date()),
    "last_price": prices.apply(lambda s: s.last_valid_index().date()),
    "observations": prices.notna().sum(),
    "missing": prices.isna().sum(),
})

,first_price,last_price,observations,missing
SYN_MARKET,2015-01-02,2024-12-31,2608,0
SYN_US_EQUITY,2015-01-02,2024-12-31,2608,0
SYN_TECH_GROWTH,2015-01-02,2024-12-31,2608,0
SYN_INTL_EQUITY,2015-01-02,2024-12-31,2601,7
SYN_EM_EQUITY,2017-03-01,2024-12-31,2045,563
SYN_TREASURY,2015-01-02,2024-12-31,2608,0
SYN_CORP_BOND,2015-01-02,2024-12-31,2608,0
SYN_GOLD,2015-01-02,2024-12-31,2608,0
SYN_REIT,2015-01-02,2024-12-31,2608,0


## 2. A balanced portfolio vs the market
A five-asset mix, rebalanced monthly, with a 2% risk-free rate.

In [3]:
weights = {"SYN_US_EQUITY": 0.35, "SYN_INTL_EQUITY": 0.15, "SYN_TREASURY": 0.25,
           "SYN_CORP_BOND": 0.15, "SYN_GOLD": 0.10}
cfg = AnalysisConfig(assets=tuple(weights), weights=weights, benchmark="SYN_MARKET",
                     rebalance="monthly", risk_free_rate=0.02, confidence=0.95)
res = run_analysis(prices, cfg)
print(f"Sample: {res.quality.effective_start:%Y-%m-%d} -> {res.quality.effective_end:%Y-%m-%d}, "
      f"{res.quality.n_return_periods} daily returns, {res.quality.rows_dropped} dates dropped")
res.summary.loc[["cumulative_return", "annualized_return", "annualized_volatility", "sharpe_ratio",
                 "max_drawdown", "historical_var", "historical_es"]]

Sample: 2015-01-02 -> 2024-12-31, 2600 daily returns, 7 dates dropped


,Portfolio,Benchmark
cumulative_return,0.8418,1.3829
annualized_return,0.0610,0.0878
annualized_volatility,0.0858,0.1651
sharpe_ratio,0.5019,0.4724
max_drawdown,-0.2033,-0.3408
historical_var,0.0081,0.0162
historical_es,0.0120,0.0238


In [4]:
pd.Series(res.relative, name="relative to SYN_MARKET")

excess_cumulative_return   -0.5411
excess_annualized_return   -0.0268
tracking_error              0.0922
information_ratio          -0.3787
beta                        0.4792
correlation                 0.9218
jensens_alpha               0.0057
Name: relative to SYN_MARKET, dtype: float64

## 3. Does the rebalancing assumption matter?
The same target weights under each rebalancing rule. Buy-and-hold lets winners grow, so
the portfolio becomes more concentrated over time. Daily rebalancing trades every day.
Turnover is one-way, summed over the sample.

In [5]:
rets = compute_returns(res.prices[list(weights)])
rows = {}
for rule in ["daily", "monthly", "quarterly", "annually", "none"]:
    pr = construct_portfolio(rets, weights, rule)
    r = pr.returns
    rows[rule] = {
        "CAGR": annualized_return(r), "volatility": annualized_volatility(r),
        "sharpe (rf 2%)": sharpe_ratio(r, 0.02), "max drawdown": max_drawdown(r),
        "rebalances": len(pr.turnover), "total turnover": pr.turnover.sum(),
        "end weight US equity": pr.end_weights["SYN_US_EQUITY"],
    }
pd.DataFrame(rows).T

,CAGR,volatility,sharpe (rf 2%),max drawdown,rebalances,total turnover,end weight US equity
daily,0.0602,0.0860,0.4924,-0.2037,"2,599.0000",6.0004,0.3514
monthly,0.0610,0.0858,0.5019,-0.2033,119.0000,1.4422,0.3581
quarterly,0.0606,0.0862,0.4958,-0.2087,39.0000,0.8331,0.3635
annually,0.0626,0.0860,0.5183,-0.2081,9.0000,0.4413,0.4156
none,0.0604,0.0979,0.4453,-0.2287,0.0000,0.0000,0.4855


## 4. Historical vs Gaussian VaR
If the Gaussian VaR is below the historical VaR at 99%, the normal distribution understates
tail risk. Excess kurtosis > 0 measures how much fatter the tails are.

In [6]:
port = res.portfolio_returns
print(f"Skewness {port.skew():.3f}, excess kurtosis {port.kurt():.3f}")
var_es_table(port, (0.90, 0.95, 0.975, 0.99), portfolio_value=1_000_000)

Skewness -0.067, excess kurtosis 2.486


,tail_observations,historical_var,historical_es,historical_var_amount,historical_es_amount,gaussian_var,gaussian_es
confidence,,,,,,,
0.9000,260,0.0061,0.0095,"6,075.3050","9,512.9435",0.0067,0.0092
0.9500,130,0.0081,0.0120,"8,099.9446","12,047.0820",0.0086,0.0109
0.9750,65,0.0111,0.0147,"11,081.4897","14,739.7905",0.0103,0.0124
0.9900,26,0.0141,0.0182,"14,075.8359","18,174.6291",0.0123,0.0142


## 5. Verifying the worked example
The 20-return example from `docs/METHODOLOGY.md`, recomputed with the library functions.

In [7]:
example = pd.Series([0.005, -0.012, 0.003, 0.021, -0.004, -0.030, 0.008, 0.011, -0.007, 0.002,
                     -0.022, 0.009, 0.015, -0.001, 0.006, -0.018, 0.004, 0.010, -0.009, 0.007])
for c in (0.90, 0.95):
    print(f"{c:.0%}: VaR = {historical_var(example, c):.2%}, ES = {historical_expected_shortfall(example, c):.2%}")
assert round(historical_var(example, 0.90), 6) == 0.022
assert round(historical_expected_shortfall(example, 0.90), 6) == 0.026

90%: VaR = 2.20%, ES = 2.60%
95%: VaR = 3.00%, ES = 3.00%


## 6. Out-of-sample VaR backtest
Each day's 95% VaR comes from the previous 252 returns. About 5% of days should breach it.
The Kupiec test checks whether the observed breach rate is consistent with 5%.

In [8]:
bt = res.backtest
pd.Series({"forecasts": bt.n_forecasts, "exceptions": bt.n_exceptions,
           "expected": bt.expected_exceptions, "exception rate": bt.exception_rate,
           "Kupiec LR": bt.kupiec_lr, "p-value": bt.kupiec_p_value})

forecasts        2,348.0000
exceptions         118.0000
expected           117.4000
exception rate       0.0503
Kupiec LR            0.0032
p-value              0.9547
dtype: float64

In [9]:
# Exceptions per calendar year: about 13 are expected in a full year at 95% (0.05 x ~260 days).
# The Kupiec test checks the overall frequency only, not whether breaches cluster in time.
bt.exceptions.groupby(bt.exceptions.index.year).agg(["sum", "count"]).rename(
    columns={"sum": "exceptions", "count": "days"})

,exceptions,days
Date,,
2015,0,7
2016,12,260
2017,16,259
2018,16,260
2019,10,260
2020,16,262
2021,11,261
2022,17,259
2023,9,259


## 7. Risk across market regimes
Days are classified by the benchmark's volatility over the *previous* 21 days.

In [10]:
res.regimes

,days,share_of_days,annualized_mean_return,annualized_volatility,historical_var,historical_es,worst_period
regime,,,,,,,
Calm,1338,0.5188,0.0482,0.0826,0.0080,0.0117,-0.0288
Normal,1129,0.4378,0.0922,0.0825,0.0075,0.0113,-0.0229
Stressed,112,0.0434,-0.0965,0.1410,0.0140,0.0203,-0.0287


In [11]:
res.calendar_years

,observations,partial_year,return,volatility,sharpe_ratio,max_drawdown,historical_var,worst_period
year,,,,,,,,
2015,259,False,0.2488,0.0756,2.6373,-0.0290,0.0072,-0.0218
2016,260,False,0.0776,0.0903,0.6277,-0.0844,0.0074,-0.0288
2017,259,False,0.1264,0.0778,1.2730,-0.0850,0.0065,-0.0170
2018,260,False,-0.0784,0.0770,-1.2459,-0.1317,0.0079,-0.0198
2019,260,False,0.2043,0.0740,2.2058,-0.0385,0.0066,-0.0143
2020,262,False,0.0503,0.1156,0.2945,-0.0850,0.0109,-0.0287
2021,261,False,0.0949,0.0785,0.9018,-0.0493,0.0068,-0.0162
2022,259,False,-0.0972,0.1028,-1.1092,-0.1203,0.0112,-0.0229
2023,259,False,-0.0585,0.0775,-0.9732,-0.1149,0.0086,-0.0178


## 8. Takeaways

- **Rebalancing is a modelling choice with visible consequences.** Without rebalancing, the
  equity weight drifts well above its 35% target and volatility rises (section 3). Frequent
  rebalancing controls drift at the cost of turnover.
- **Diversification works through correlation.** See `res.risk_contributions` below. US equity
  carries far more of the risk than of the capital, while the treasury sleeve's contribution is
  close to zero or negative, so it acts as a hedge.
- **Tails are fatter than normal.** Historical VaR is below the Gaussian figure at 90% but
  above it at 97.5% and 99% (section 4), which is the signature of fat tails.
- **The backtest passes on frequency.** Breaches are close to the expected 5% overall and
  spread fairly evenly across years in this synthetic sample. Real market data usually shows
  stronger clustering, which the Kupiec test alone would not detect.

All of this describes one synthetic history and is not a forecast.

In [12]:
res.risk_contributions

,weight,asset_volatility,risk_contribution,pct_of_risk
SYN_US_EQUITY,0.3500,0.1713,0.0561,0.6531
SYN_INTL_EQUITY,0.1500,0.1597,0.0209,0.2429
SYN_TREASURY,0.2500,0.0612,-0.0001,-0.0010
SYN_CORP_BOND,0.1500,0.0516,0.0050,0.0586
SYN_GOLD,0.1000,0.1443,0.0040,0.0465
